**Dự đoán quốc gia vô địch World Cup**

**I. Nạp dataset**

In [275]:
from pathlib import Path
import pandas as pd

def save_csv(dataframe, file_name, output_dir='../loc'):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / file_name
    dataframe.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f'Đã lưu: {file_path}')

df_games = pd.read_csv('../dataset/du_lieu_tho/results.csv')
df_games2026 = pd.read_csv('../dataset/du_lieu_tho/match_prediction_features.csv')
df_games2026 = df_games2026.rename(columns={
    'home_team_name': 'home_team',
    'away_team_name': 'away_team',
    'venue_city': 'city',
    'venue_country': 'country',
})
TEAM_NAME_ALIASES = {
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'IR Iran': 'Iran',
    'USA': 'United States',
    'Congo DR': 'DR Congo',
    "Côte d'Ivoire": 'Ivory Coast',
    'Cabo Verde': 'Cape Verde',
}
for team_column in ('home_team', 'away_team'):
    df_games[team_column] = df_games[team_column].replace(TEAM_NAME_ALIASES)
    df_games2026[team_column] = df_games2026[team_column].replace(TEAM_NAME_ALIASES)
df_games2026['tournament'] = 'FIFA World Cup'
df_games2026['country'] = df_games2026['country'].map({
    'CAN': 'Canada',
    'MEX': 'Mexico',
    'USA': 'United States',
})
df_games2026['neutral'] = ~(
    df_games2026['home_is_host'].astype(bool)
    | df_games2026['away_is_host'].astype(bool)
)
df_games2026 = df_games2026[df_games.columns]

df_group_stage = pd.read_csv('../dataset/mo_phong/group_stages.csv', sep=';')
df_knockout_fixtures = pd.read_csv('../dataset/mo_phong/fixtures_knockout_wc2026.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

Thống Kê các trận đâu truớc 2026

In [276]:
print(df_games.shape)
df_games.head()

(49945, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


Thống Kê các trận đâu trong 2026

In [277]:
print(df_games2026.shape)
df_games2026.head()

(104, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-06-11,Mexico,South Africa,2,0,FIFA World Cup,Mexico City,Mexico,False
1,2026-06-11,South Korea,Czech Republic,2,1,FIFA World Cup,Zapopan,Mexico,True
2,2026-06-12,Canada,Bosnia and Herzegovina,1,1,FIFA World Cup,Toronto,Canada,False
3,2026-06-12,United States,Paraguay,4,1,FIFA World Cup,Inglewood,United States,False
4,2026-06-13,Qatar,Switzerland,1,1,FIFA World Cup,Santa Clara,United States,True


**II. Lọc dữ liệu**

**Truớc WC 2026**

In [278]:
print(df_games.shape)
df_games.head()

(49945, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [279]:
df_games.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49940,2026-10-06,Belize,French Guiana,0,1,CONCACAF Nations League,Belmopan,Belize,False
49941,2026-10-06,Turks and Caicos Islands,Montserrat,1,4,CONCACAF Nations League,Providenciales,Turks and Caicos Islands,False
49942,2026-10-06,Antigua and Barbuda,Aruba,3,2,CONCACAF Nations League,North Sound,Antigua and Barbuda,False
49943,2026-10-06,Saint Martin,Bahamas,6,0,CONCACAF Nations League,St Croix,United States Virgin Islands,True
49944,2026-10-06,Saudi Arabia,United Arab Emirates,2,0,Gulf Cup,Jeddah,Saudi Arabia,False


In [280]:
df_games['tournament'].value_counts().head()

tournament
Friendly                                18263
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2387
FIFA World Cup                           1068
Name: count, dtype: int64

**Giữ trận từ ngày khai mạc WC 2018 đến trước WC 2026, bỏ CONIFA**

Phong độ mỗi đội dùng trung bình bàn ghi và thủng trong tối đa 5 trận trước đó. Các trận cùng ngày không dùng kết quả của nhau.

In [281]:
from collections import defaultdict, deque
from itertools import groupby

FORM_COLUMNS = [
    'home_form_goals_scored_last5', 'home_form_goals_conceded_last5',
    'away_form_goals_scored_last5', 'away_form_goals_conceded_last5',
]
def add_pre_match_form(history_source_df, history_target_df, world_cup_df):
    history_rows = [
        (index, row.to_dict())
        for index, row in history_source_df.iterrows()
        if str(row['date']) < '2026-06-11'
        and 'conifa' not in str(row['tournament']).casefold()
    ]
    world_cup_rows = world_cup_df.to_dict('records')
    events = [
        (str(row['date']), 'history', index, row)
        for index, row in history_rows
    ] + [
        (str(row['date']), 'world_cup', index, row)
        for index, row in enumerate(world_cup_rows)
    ]
    events.sort(key=lambda event: (event[0], event[1], event[2]))
    team_form = defaultdict(lambda: deque(maxlen=5))
    total_goals = 0.0
    team_match_count = 0
    history_form = {}
    world_cup_form = [None] * len(world_cup_rows)

    for day, day_events in groupby(events, key=lambda event: event[0]):
        day_events = list(day_events)
        for _, source, index, row in day_events:
            home_history = team_form[row['home_team']]
            away_history = team_form[row['away_team']]
            fallback = total_goals / team_match_count if team_match_count else 1.3
            features = {
                'home_form_goals_scored_last5': (
                    sum(match[0] for match in home_history) / len(home_history)
                    if home_history else fallback
                ),
                'home_form_goals_conceded_last5': (
                    sum(match[1] for match in home_history) / len(home_history)
                    if home_history else fallback
                ),
                'away_form_goals_scored_last5': (
                    sum(match[0] for match in away_history) / len(away_history)
                    if away_history else fallback
                ),
                'away_form_goals_conceded_last5': (
                    sum(match[1] for match in away_history) / len(away_history)
                    if away_history else fallback
                ),
            }
            if source == 'history':
                history_form[index] = features
            else:
                world_cup_form[index] = features

        for _, _, _, row in day_events:
            if 'conifa' in str(row.get('tournament', '')).casefold():
                continue
            home_goals = float(row['home_score'])
            away_goals = float(row['away_score'])
            team_form[row['home_team']].append((home_goals, away_goals))
            team_form[row['away_team']].append((away_goals, home_goals))
            total_goals += home_goals + away_goals
            team_match_count += 2

    history_result = history_target_df.copy()
    for column in FORM_COLUMNS:
        history_result[column] = [history_form[index][column] for index in history_result.index]
    world_cup_result = world_cup_df.copy()
    for column in FORM_COLUMNS:
        world_cup_result[column] = [row[column] for row in world_cup_form]

    return history_result, world_cup_result

df_history_all = df_games.copy()
dates = pd.to_datetime(df_games['date'])
is_conifa = df_games['tournament'].str.contains('CONIFA', case=False, na=False)

df_games = df_games.loc[
    (dates >= pd.Timestamp('2018-06-14'))
    & (dates < pd.Timestamp('2026-06-11'))
    & ~is_conifa
].copy()

df_games, df_games2026 = add_pre_match_form(df_history_all, df_games, df_games2026)

save_csv(df_games, 'matches_truoc_wc_2026_da_loc.csv')

print(df_games.shape)
df_games.head()

Đã lưu: ..\loc\matches_truoc_wc_2026_da_loc.csv
(7768, 13)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_form_goals_scored_last5,home_form_goals_conceded_last5,away_form_goals_scored_last5,away_form_goals_conceded_last5
41657,2018-06-14,Russia,Saudi Arabia,5,0,FIFA World Cup,Moscow,Russia,False,1.0,2.2,1.2,1.4
41658,2018-06-15,Egypt,Uruguay,0,1,FIFA World Cup,Ekaterinburg,Russia,True,0.4,1.4,1.4,0.4
41659,2018-06-15,Morocco,Iran,0,1,FIFA World Cup,Saint Petersburg,Russia,True,1.8,0.6,1.0,0.8
41660,2018-06-15,Portugal,Spain,3,3,FIFA World Cup,Sochi,Russia,True,1.4,1.2,2.4,1.2
41661,2018-06-16,France,Australia,2,1,FIFA World Cup,Kazan,Russia,True,2.2,1.2,2.0,1.2


In [282]:
print(df_games2026.shape)
df_games2026.head()

(104, 13)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_form_goals_scored_last5,home_form_goals_conceded_last5,away_form_goals_scored_last5,away_form_goals_conceded_last5
0,2026-06-11,Mexico,South Africa,2,0,FIFA World Cup,Mexico City,Mexico,False,1.8,0.4,0.8,1.0
1,2026-06-11,South Korea,Czech Republic,2,1,FIFA World Cup,Zapopan,Mexico,True,1.4,1.0,3.4,1.2
2,2026-06-12,Canada,Bosnia and Herzegovina,1,1,FIFA World Cup,Toronto,Canada,False,1.4,0.6,0.8,0.8
3,2026-06-12,United States,Paraguay,4,1,FIFA World Cup,Inglewood,United States,False,2.2,2.4,1.8,1.0
4,2026-06-13,Qatar,Switzerland,1,1,FIFA World Cup,Santa Clara,United States,True,0.2,1.2,1.8,1.4


In [283]:
df_games2026.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_form_goals_scored_last5,home_form_goals_conceded_last5,away_form_goals_scored_last5,away_form_goals_conceded_last5
99,2026-07-12,Argentina,Switzerland,3,1,FIFA World Cup,Kansas City,United States,True,2.8,1.0,1.8,0.6
100,2026-07-14,France,Spain,0,2,FIFA World Cup,Arlington,United States,True,2.6,0.2,2.2,0.2
101,2026-07-15,England,Argentina,1,2,FIFA World Cup,Atlanta,United States,True,1.8,0.8,2.8,1.2
102,2026-07-18,France,England,4,6,FIFA World Cup,Miami Gardens,United States,True,2.0,0.6,2.0,1.2
103,2026-07-19,Spain,Argentina,1,0,FIFA World Cup,East Rutherford,United States,True,1.8,0.2,2.8,1.4


## III. Tính chỉ số Elo

Elo được tính tuần tự trên toàn bộ lịch sử từ 1930, để mỗi đội có đủ trận cho Elo hội tụ trước khi bước vào cửa sổ huấn luyện. Tập huấn luyện vẫn chỉ lấy trận từ 14/06/2018.

**Vì sao phải tính Elo từ 1930:** nếu khởi tạo mọi đội bằng 1500 đúng tại ngày khai mạc WC 2018 thì trong khoảng 1–2 năm đầu Elo gần như vô nghĩa (đội mạnh và đội yếu đều bắt đầu từ 1500). Điều đó làm hỏng một phần tư tập huấn luyện.

**Lưu ý quan trọng về tập test:** Elo của 104 trận WC 2026 cũng phải được cập nhật lần lượt qua từng trận, giống hệt cách tính lúc huấn luyện. Nếu gán cùng một giá trị Elo (lấy trước giải) cho cả 104 trận thì đặc trưng của tập test sẽ khác phân tích so với tập train — đây là lỗi dễ mắc nhất khi dự đoán một giải đấu.

In [284]:
from collections import defaultdict

TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa Am\u00e9rica', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa Am\u00e9rica qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Am\u00edlcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

TIER_5_FRIENDLY = {'Friendly', 'FIFA Series', 'CONCACAF Series'}


def get_k_factor(tournament):
    """K càng lớn thì Elo phản ứng càng nhanh với kết quả mới.

    Trận quan trọng (World Cup) được cập nhật mạnh hơn trận giao hữu.
    """
    if tournament in TIER_1_WORLD_CUP:
        return 60
    if tournament in TIER_2_CONTINENTAL:
        return 50
    if tournament in TIER_3_QUALIFIERS_NATIONS:
        return 40
    if tournament in TIER_4_REGIONAL:
        return 30
    if tournament in TIER_5_FRIENDLY:
        return 20
    return 15


HOME_ADVANTAGE = 100  # lợi thế sân nhà quy đổi ra điểm Elo


def expected_score(home_elo, away_elo, home_advantage=HOME_ADVANTAGE):
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))


def parse_neutral(value):
    return str(value).strip().casefold() in {'true', '1', 'yes'}


def calculate_elo(df, initial_elo=1500):
    """Tính Elo tuần tự, trả về Elo TRƯỚC mỗi trận (không dùng kết quả trận đó)."""
    source_records = df.to_dict("records") if hasattr(df, "to_dict") else [dict(r) for r in df]
    scored_records = []
    for record in source_records:
        if 'conifa' in str(record.get("tournament", "")).casefold():
            continue
        scored_records.append((
            record,
            float(record["home_score"]),
            float(record["away_score"]),
        ))

    # Sắp xếp theo datetime thay vì theo chuỗi: so sánh chuỗi chỉ đúng khi
    # mọi ngày đều ở định dạng YYYY-MM-DD, đổi định dạng là sai thứ tự ngay.
    scored_records.sort(key=lambda item: pd.to_datetime(item[0]["date"]))
    elo_dict = defaultdict(lambda: float(initial_elo))
    elo_history = []

    for record, home_score, away_score in scored_records:
        home, away = record["home_team"], record["away_team"]
        home_elo, away_elo = elo_dict[home], elo_dict[away]

        home_advantage = 0 if parse_neutral(record.get("neutral", False)) else HOME_ADVANTAGE
        exp_home = expected_score(home_elo, away_elo, home_advantage)
        exp_away = 1 - exp_home

        if home_score > away_score:
            actual_home, actual_away = 1.0, 0.0
        elif home_score < away_score:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5

        # Thắng càng đậm thì Elo đổi càng nhiều.
        goal_diff = abs(home_score - away_score)
        if goal_diff <= 1:
            goal_diff_multiplier = 1.0
        elif goal_diff == 2:
            goal_diff_multiplier = 1.5
        else:
            goal_diff_multiplier = (11 + goal_diff) / 8

        k = get_k_factor(record["tournament"])
        elo_dict[home] = home_elo + k * goal_diff_multiplier * (actual_home - exp_home)
        elo_dict[away] = away_elo + k * goal_diff_multiplier * (actual_away - exp_away)

        record["home_elo_pre"] = home_elo
        record["away_elo_pre"] = away_elo
        elo_history.extend([
            (record["date"], home, elo_dict[home]),
            (record["date"], away, elo_dict[away]),
        ])

    return [item[0] for item in scored_records], elo_dict, elo_history


### Chạy tính Elo và tạo đặc trưng trận đấu

Một lượt chạy duy nhất trên trục thời gian liên tục từ 1930 đến hết WC 2026, để Elo của các trận WC 2026 cũng được cập nhật qua từng trận. Sau đó tách thành hai tập: huấn luyện (14/06/2018 – 10/06/2026) và kiểm tra (104 trận WC 2026).

Bảy feature đầu vào: Elo hai đội trước trận, cờ sân trung lập, và 4 chỉ số bàn ghi/thủng trung bình trong tối đa 5 trận trước đó.

In [ ]:
ELO_START = pd.Timestamp('1930-01-01')
TRAIN_START = pd.Timestamp('2018-06-14')
WC_START = pd.Timestamp('2026-06-11')
WC_END = pd.Timestamp('2026-07-20')

BASE_COLUMNS = [
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'tournament', 'city', 'country', 'neutral',
]
FEATURE_COLUMNS = [
    'home_elo_pre', 'away_elo_pre',
    'home_form_goals_scored_last5', 'home_form_goals_conceded_last5',
    'away_form_goals_scored_last5', 'away_form_goals_conceded_last5',
]

# Ba khúc nối tiếp nhau trên cùng một trục thời gian:
#   (1) 1930 -> 13/06/2018: chỉ để Elo "nóng máy", không dùng huấn luyện
#   (2) 14/06/2018 -> 10/06/2026: tập huấn luyện, đã có form
#   (3) 104 trận WC 2026: tập kiểm tra, Elo vẫn cập nhật qua từng trận
history_dates = pd.to_datetime(df_history_all['date'])
warmup_part = df_history_all.loc[
    (history_dates >= ELO_START) & (history_dates < TRAIN_START)
].copy()
warmup_part['_is_world_cup_2026'] = False

training_part = df_games.copy()
training_part['_is_world_cup_2026'] = False

world_cup_part = df_games2026.copy()
world_cup_part['_is_world_cup_2026'] = True

for part in (warmup_part, training_part, world_cup_part):
    part['_date'] = pd.to_datetime(part['date'])

combined_source = pd.concat(
    [warmup_part, training_part, world_cup_part], ignore_index=True
).sort_values('_date', kind='stable').reset_index(drop=True)

matches_with_elo, _, elo_history = calculate_elo(combined_source)
print(f"Tính Elo trên {len(matches_with_elo):,} trận, từ 1930 đến hết WC 2026")

with_elo = pd.DataFrame(matches_with_elo)

df_match_features = with_elo.loc[
    (~with_elo['_is_world_cup_2026']) & (with_elo['_date'] >= TRAIN_START)
].copy()
df_match_features = (
    df_match_features[BASE_COLUMNS + FEATURE_COLUMNS]
    .sort_values('date').reset_index(drop=True)
)

df_games2026 = with_elo.loc[with_elo['_is_world_cup_2026']].copy()
df_games2026 = (
    df_games2026[BASE_COLUMNS + FEATURE_COLUMNS]
    .sort_values('date').reset_index(drop=True)
)

print(f"Tập huấn luyện: {len(df_match_features):,} trận "
      f"({df_match_features['date'].min()} đến {df_match_features['date'].max()})")
print(f"Tập kiểm tra WC 2026: {len(df_games2026):,} trận "
      f"({df_games2026['date'].min()} đến {df_games2026['date'].max()})")

# --- Kiểm tra dữ liệu trước khi lưu ---
assert not df_match_features[FEATURE_COLUMNS].isna().any().any(), 'Train còn NaN'
assert not df_games2026[FEATURE_COLUMNS].isna().any().any(), 'Test còn NaN'
assert len(df_match_features) == len(df_games), 'Số trận huấn luyện bị lệch'
assert len(df_games2026) == len(world_cup_part), 'Số trận test bị lệch'

# Elo phải thay đổi giữa các trận trong cùng một giải, không được đóng băng.
print(f"home_elo_pre có {df_games2026['home_elo_pre'].nunique()} giá trị khác nhau "
      f"trên {len(df_games2026)} trận, {df_games2026['home_team'].nunique()} đội đá sân nhà.")

# Đội chưa từng đá trận nào trước WC 2026 sẽ nhận Elo khởi tạo 1500.
_, elos_before_world_cup, _ = calculate_elo(
    combined_source.loc[combined_source['_date'] < WC_START]
)
world_cup_teams = set(df_games2026['home_team']) | set(df_games2026['away_team'])
teams_without_history = sorted(t for t in world_cup_teams if t not in elos_before_world_cup)
if teams_without_history:
    print(f"Đội chưa có Elo lịch sử (dùng mặc định 1500): {teams_without_history}")
else:
    print("Mọi đội dự WC 2026 đều đã có Elo lịch sử.")

save_csv(df_match_features, 'matches_truoc_wc_2026_elo_features.csv')
save_csv(df_games2026, 'wc_2026_da_loc.csv')

# Gộp SAU khi đã có Elo, để file gộp không thiếu cột đặc trưng.
df_all_matches = (
    pd.concat([df_match_features, df_games2026], ignore_index=True)
    .sort_values('date').reset_index(drop=True)
)
save_csv(df_all_matches, 'matches_truoc_va_sau_wc_2026.csv')

df_elo = pd.DataFrame(elo_history, columns=['date', 'team', 'elo_after'])
df_elo['date'] = pd.to_datetime(df_elo['date'])
print(f"df_elo shape: {df_elo.shape}")

sorted_elos = sorted(elos_before_world_cup.items(), key=lambda x: x[1], reverse=True)
print("\nTop 20 đội có Elo cao nhất ngay trước WC 2026:")
for i, (team, elo) in enumerate(sorted_elos[:20], 1):
    print(f"{i:2d}. {team:<25}: {elo:.2f}")


### Kiểm tra 10 đội có Elo thấp nhất

Hiển thị các đội có Elo thấp nhất ngay trước WC 2026 để rà soát kết quả tính toán.

In [ ]:
bottom_10 = sorted_elos[-10:][::-1]
print("10 đội có Elo thấp nhất ngay trước WC 2026:")
for i, (team, elo) in enumerate(bottom_10, 1):
    print(f"{i:2d}. {team:<25}: {elo:.2f}")


10 đội có Elo thấp nhất ngay trước WC 2026:
 1. Macau                    : 855.90
 2. Bhutan                   : 869.03
 3. Anguilla                 : 898.65
 4. San Marino               : 902.40
 5. Brunei                   : 905.85
 6. Timor-Leste              : 914.74
 7. Laos                     : 933.83
 8. United States Virgin Islands: 936.30
 9. American Samoa           : 950.53
10. Liechtenstein            : 963.37


### Bảng 7 feature cho các trận có đội thuộc top 10 Elo

Hiển thị 7 feature đầu vào mô hình cho các trận huấn luyện có ít nhất một đội thuộc top 10 Elo.

In [ ]:
top_10_teams = {team for team, _ in sorted_elos[:10]}
model_feature_columns = [
    'home_elo_pre', 'away_elo_pre', 'neutral',
    'home_form_goals_scored_last5', 'home_form_goals_conceded_last5',
    'away_form_goals_scored_last5', 'away_form_goals_conceded_last5',
]
top_10_match_features = df_match_features.loc[
    df_match_features['home_team'].isin(top_10_teams)
    | df_match_features['away_team'].isin(top_10_teams),
    ['date', 'home_team', 'away_team', *model_feature_columns],
].sort_values('date').reset_index(drop=True)

print(f"Số trận huấn luyện có ít nhất một đội trong top 10 Elo: {len(top_10_match_features):,}")
display(top_10_match_features.tail(10))


Số trận huấn luyện có ít nhất một đội trong top 10 Elo: 871


,date,home_team,away_team,home_elo_pre,away_elo_pre,neutral,home_form_goals_scored_last5,home_form_goals_conceded_last5,away_form_goals_scored_last5,away_form_goals_conceded_last5
861,2026-06-06,Brazil,Egypt,2068.204455,1818.051774,True,2.6,1.2,1.0,0.2
862,2026-06-06,Argentina,Honduras,2180.111198,1708.014526,True,3.2,0.2,1.0,0.8
863,2026-06-07,Jordan,Colombia,1775.530730,2054.249733,True,1.6,2.2,2.0,1.4
864,2026-06-07,Ecuador,Guatemala,2020.654424,1645.721040,True,1.2,0.6,1.4,3.2
865,2026-06-08,Netherlands,Uzbekistan,2000.385274,1813.659691,True,1.6,0.8,1.0,1.0
866,2026-06-08,Peru,Spain,1770.435979,2211.598958,True,1.2,1.6,2.0,0.6
867,2026-06-08,France,Northern Ireland,2118.776284,1657.972358,False,2.6,1.0,0.6,0.8
868,2026-06-09,Argentina,Iceland,2181.969469,1625.443705,True,3.4,0.2,0.6,2.0
869,2026-06-10,Portugal,Nigeria,2037.478599,1872.506852,False,2.6,0.8,2.2,1.0
870,2026-06-10,England,Costa Rica,2081.903687,1749.048978,True,1.2,0.4,0.6,2.2
